# Extended thinking

Extended thinking lets Claude emit a hidden-ish reasoning phase before the final answer.
You see the thinking tokens (billed) but the model treats them as working memory, not output.

## How to enable

`chat(..., thinking=True, thinking_budget=N)`.
- `thinking_budget` is the minimum tokens reserved for the thinking phase. Must be ≥1024.
- `max_tokens` must be `> thinking_budget` (budget is included inside `max_tokens`).
- When to use: multistep problems where the first-token answer is likely wrong without reflection.
  Use prompt evals to decide, not intuition.

## Response shape

`response.content` becomes a list of blocks: one or more `thinking` blocks followed by `text` blocks.
Each `thinking` block carries a cryptographic `signature` — a tamper-check tied to the thinking text.
If you reuse the message in a follow-up turn, keep the thinking block exactly as returned;
any edit invalidates the signature and the next request will be rejected.

## Redacted thinking

Sometimes Claude's internal safety checks flag the thinking as potentially unsafe
and return a `redacted_thinking` block instead of a plain `thinking` block.
The content is encrypted and opaque to you.
The defense: because you cannot read it, you cannot inject instructions based on what you *think* Claude reasoned;
and because it is signed, you cannot modify it undetected either.
You can force a redacted response in testing with the magic trigger string Anthropic publishes, used in the demo below.

Model: `claude-sonnet-4-6`.


In [1]:
from chat_tools import add_user_message, chat
from models import MODEL_SONNET

# Magic string that forces the API to return a `redacted_thinking` block (for testing only).
THINKING_TEST_STR = (
    "ANTHROPIC_MAGIC_STRING_TRIGGER_REDACTED_THINKING_46C9A13E193C177646C7398A98432ECCCE4C1253D5E2D82641AC0E52CC2876CB"
)

In [2]:
# Normal thinking: Claude reasons before answering a multistep question.
question = (
    "If a train leaves Madrid at 09:00 going 120 km/h "
    "and another leaves Barcelona at 10:30 going 90 km/h "
    "on the same single-track 620 km line, "
    "where and when do they meet?"
)

messages = []
add_user_message(messages, question)
response = chat(messages, model=MODEL_SONNET, max_tokens=4000, thinking=True, thinking_budget=1024)

for block in response.content:
    print(f"--- {block.type} ---")
    if block.type == "thinking":
        print(block.thinking[:400], "..." if len(block.thinking) > 400 else "")
    elif block.type == "text":
        print(block.text)
    else:
        print(str(block)[:200])

--- thinking ---
Let me set up the problem. The distance between Madrid and Barcelona is 620 km.

Train 1: Leaves Madrid at 09:00, speed = 120 km/h
Train 2: Leaves Barcelona at 10:30, speed = 90 km/h

They travel toward each other on the same track.

By 10:30 (when Train 2 departs), Train 1 has already traveled:
1.5 hours × 120 km/h = 180 km

Remaining distance between them at 10:30: 620 - 180 = 440 km

They appro ...
--- text ---
## Setting Up the Problem

**Given information:**
- Track length: 620 km (Madrid → Barcelona)
- Train M (Madrid): departs 09:00 at 120 km/h
- Train B (Barcelona): departs 10:30 at 90 km/h, traveling toward Madrid

---

## Step 1: Head Start by Train M

Train M travels **1.5 hours alone** before Train B departs:

$$d_{headstart} = 120 \times 1.5 = 180 \text{ km}$$

**Remaining gap at 10:30:** 620 − 180 = **440 km**

---

## Step 2: Time Until They Meet (after 10:30)

The trains close the gap at their **combined speed**:

$$t = \frac{440}{120 + 90} = \frac{440}

In [3]:
# Redacted thinking: force the model to return an encrypted thinking block.
messages = []
add_user_message(messages, THINKING_TEST_STR)
response = chat(messages, model=MODEL_SONNET, max_tokens=4000, thinking=True, thinking_budget=1024)

for block in response.content:
    print(f"--- {block.type} ---")
    if block.type == "redacted_thinking":
        print(f"opaque blob of {len(block.data)} chars")
    elif block.type == "text":
        print(block.text)
    else:
        print(str(block)[:200])

--- redacted_thinking ---
opaque blob of 512 chars
--- text ---
It looks like you've sent a test or trigger string — possibly related to Anthropic's internal systems or a prompt injection attempt.

I don't have any special behavior tied to that string. I'm happy to help you with whatever you actually need!

What can I do for you?
